# CropQuery — Main Notebook

Run every cell top-to-bottom at the start of each Kaggle session.

| # | Section |
|---|--------------------------------------------|
| 1 | Install dependencies |
| 2 | Set secrets |
| 3 | Write module files |
| 4 | Verify paths + write class_names.json |
| 5 | Build knowledge base |
| 6 | Load pipeline |
| 7 | FastAPI app definition |
| 8 | Start uvicorn server |
| 9 | Start ngrok tunnel |
|10 | Smoke test |
|11 | Launch Gradio UI |

## 1 · Install dependencies

In [1]:
!pip install -q \
    gradio \
    chromadb \
    faiss-cpu \
    sentence-transformers \
    accelerate \
    bitsandbytes \
    deep-translator \
    timm \
    fastapi \
    uvicorn \
    pyngrok \
    nest_asyncio \
    python-multipart

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 18.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 94.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 64.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.0 MB/s eta 0:00:00


## 2 · Set secrets

In [2]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.makedirs("/kaggle/working/cropquery/", exist_ok=True)
print("HF_TOKEN set. Working dir ready.")

HF_TOKEN set. Working dir ready.


## 3 · LLaVA Caching

In [3]:
import os
from pathlib import Path

LLAVA_CACHE_DIR = "/kaggle/working/llava_cache"
LLAVA_MODEL_ID  = "llava-hf/llava-v1.6-mistral-7b-hf"

# If already cached in the dataset input, create a symlink so transformers
# finds it at LLAVA_CACHE_DIR without copying 15 GB.
DATASET_CACHE = "/kaggle/input/cropquerydataset/llava_cache"

if os.path.exists(DATASET_CACHE):
    if not os.path.exists(LLAVA_CACHE_DIR):
        os.symlink(DATASET_CACHE, LLAVA_CACHE_DIR)
    print(f"[Cache] LLaVA loaded from dataset cache — no download needed.")
else:
    # First-time: download and save to /kaggle/working then upload manually
    print("[Cache] Downloading LLaVA model (one-time, ~15 GB)...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=LLAVA_MODEL_ID,
        local_dir=LLAVA_CACHE_DIR,
        token=os.environ["HF_TOKEN"],
    )
    print(f"[Cache] Download complete → {LLAVA_CACHE_DIR}")
    print("[Cache] ACTION REQUIRED: Upload /kaggle/working/llava_cache as a")
    print("        new version of vitthaljauhari2/cropquerydataset, then re-run.")

[Cache] Downloading LLaVA model (one-time, ~15 GB)...


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

[Cache] Download complete → /kaggle/working/llava_cache
[Cache] ACTION REQUIRED: Upload /kaggle/working/llava_cache as a
        new version of vitthaljauhari2/cropquerydataset, then re-run.


## 3 · Write module files

Run all sub-cells. Each writes one `.py` file to `/kaggle/working/cropquery/`.

### 3a · config.py

In [4]:
%%writefile /kaggle/working/cropquery/config.py
import os
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
CHROMA_DIR = "/kaggle/working/chroma_db"

# ── Device assignment ──────────────────────────────────────────────
BACKBONE_DEVICE = "cuda:0"
LLAVA_DEVICE    = "cuda:1"

# ── Checkpoint paths ── single source of truth ─────────────────────
RUNS_DIR          = "/kaggle/input/datasets/vitthaljauhari2/cropquerydataset"
RESNET_CKPT       = os.path.join(RUNS_DIR, "resnet50_best.pth")
EFFICIENTNET_CKPT = os.path.join(RUNS_DIR, "efficientnet_best.pth")
DENSENET_CKPT     = os.path.join(RUNS_DIR, "densenet_best.pth")

# ── FAISS feature bank ─────────────────────────────────────────────
BANK_PREFIX   = os.path.join(RUNS_DIR, "feature_bank")
BANK_FEATURES = f"{BANK_PREFIX}_features.npy"
BANK_LABELS   = f"{BANK_PREFIX}_labels.npy"

# ── Class map ──────────────────────────────────────────────────────
CLASS_NAMES_PATH = os.path.join(BASE_DIR, "class_names.json")
NUM_CLASSES      = 64

# ── Vision / combiner ──────────────────────────────────────────────
IMG_SIZE             = 224
CONFIDENCE_GATE      = 80.0
FAISS_MIN_SIMILARITY = 0.65
KNN_K                = 7
TOP_K                = 3

# ── Embedding / RAG ────────────────────────────────────────────────
EMBED_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "cropquery_knowledge"
TOP_CHUNKS      = 5
TOP_K_AGROBENCH = 3
TOP_K_KISAN     = 2

# ── LLaVA ─────────────────────────────────────────────────────────
LLAVA_CACHE_DIR  = "/kaggle/working/llava_cache" 
LLAVA_MODEL_ID   = LLAVA_CACHE_DIR       
LLAVA_MAX_TOKENS = 768
LOAD_IN_4BIT     = True

# ── Translation ────────────────────────────────────────────────────
DEFAULT_LANGUAGE = "Hindi"
LANGUAGE_CODES = {
    "Hindi":     "hin_Deva",
    "Marathi":   "mar_Deva",
    "Bengali":   "ben_Beng",
    "Telugu":    "tel_Telu",
    "Tamil":     "tam_Taml",
    "Kannada":   "kan_Knda",
    "Gujarati":  "guj_Gujr",
    "Punjabi":   "pan_Guru",
    "Odia":      "ory_Orya",
    "Malayalam": "mal_Mlym",
    "English":   None,
}

# ── Weather / Location ─────────────────────────────────────────────
WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast"
GEOCODE_API_URL = "https://nominatim.openstreetmap.org/reverse"

# ── HuggingFace dataset ────────────────────────────────────────────
HF_AGROBENCH_DATASET = "vitthaljauhari/agrobench"  # single source of truth — matches Cell 5
HF_KISAN_DATASET     = "Franklin01/Dataset_prep01"

Overwriting /kaggle/working/cropquery/config.py


### 3b · vision.py

**Fixes vs old version:**
- Checkpoint paths read from `config.RESNET_CKPT / EFFICIENTNET_CKPT / DENSENET_CKPT` — not hardcoded
- `predict()` now returns `(predictions, tensor)` tuple — removes `_last_tensor` side-channel
- `gate()` now accepts tensor directly from the tuple — no hidden state

In [5]:
%%writefile /kaggle/working/cropquery/vision.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
from typing import List, Tuple, Optional, Dict
import json

from .config import (
    IMG_SIZE, CLASS_NAMES_PATH,
    RESNET_CKPT, EFFICIENTNET_CKPT, DENSENET_CKPT,
    BANK_FEATURES, BANK_LABELS, KNN_K,
    CONFIDENCE_GATE, FAISS_MIN_SIMILARITY,
)


def remap_state_dict(state_dict: dict) -> dict:
    new_sd = {}
    for k, v in state_dict.items():
        if k.endswith('num_batches_tracked'):
            continue
        if k.startswith('encoder.'):
            new_k = 'features.' + k[8:]
        elif k.startswith('head.'):
            new_k = 'classifier.' + k[5:]
        else:
            new_k = k
        new_sd[new_k] = v
    return new_sd


class ResNet50Backbone(nn.Module):
    def __init__(self, n_classes: int = 64):
        super().__init__()
        base = models.resnet50(weights=None)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(2048, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

    def extract_features(self, x):
        return self.features(x).flatten(1)


class EfficientNetB3Backbone(nn.Module):
    def __init__(self, n_classes: int = 64):
        super().__init__()
        base = models.efficientnet_b3(weights=None)
        self.features = base.features
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1536, n_classes)
        )

    def forward(self, x):
        f = F.adaptive_avg_pool2d(self.features(x), 1).flatten(1)
        return self.classifier(f)

    def extract_features(self, x):
        return F.adaptive_avg_pool2d(self.features(x), 1).flatten(1)


class DenseNet121Backbone(nn.Module):
    def __init__(self, n_classes: int = 64):
        super().__init__()
        base = models.densenet121(weights=None)
        self.features = base.features
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1024, n_classes)
        )

    def forward(self, x):
        f = F.relu(self.features(x), inplace=True)
        f = F.adaptive_avg_pool2d(f, 1).flatten(1)
        return self.classifier(f)

    def extract_features(self, x):
        f = F.relu(self.features(x), inplace=True)
        return F.adaptive_avg_pool2d(f, 1).flatten(1)


class FAISSRetriever:
    """CPU-based FAISS k-NN retriever using a pre-built feature bank."""

    def __init__(self, features_path: str, labels_path: str, k: int = 7):
        import faiss
        self.k = k
        features = np.load(features_path).astype(np.float32)
        self.labels = np.load(labels_path)
        faiss.normalize_L2(features)
        self.index = faiss.IndexFlatIP(features.shape[1])
        self.index.add(features)
        print(f"[FAISS] Loaded index: {features.shape[0]} vectors, "
              f"dim={features.shape[1]}, k={k}")

    def query(self, feature_vec: np.ndarray) -> Tuple[int, float]:
        import faiss
        vec = feature_vec.astype(np.float32).reshape(1, -1)
        faiss.normalize_L2(vec)
        distances, indices = self.index.search(vec, self.k)
        neighbour_labels = self.labels[indices[0]]
        values, counts = np.unique(neighbour_labels, return_counts=True)
        best_idx = int(values[counts.argmax()])
        top_similarity = float(distances[0][0])
        return best_idx, top_similarity


class CropVisionModel:
    """
    3-backbone ensemble: ResNet50 + EfficientNet-B3 + DenseNet121.
    Checkpoint paths come from config — not hardcoded here.
    predict() returns (predictions, img_tensor) tuple.
    gate() accepts that tensor directly — no side-channel state.
    """

    def __init__(self, device: str = "cuda:0"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        with open(CLASS_NAMES_PATH, 'r') as f:
            self.class_names = json.load(f)
        self.n_classes = len(self.class_names)

        checkpoints = {
            "resnet50":     (ResNet50Backbone,      RESNET_CKPT),
            "efficientnet": (EfficientNetB3Backbone, EFFICIENTNET_CKPT),
            "densenet":     (DenseNet121Backbone,    DENSENET_CKPT),
        }

        self.backbones: Dict[str, nn.Module] = {}
        for name, (cls, path) in checkpoints.items():
            model = cls(self.n_classes)
            ckpt = torch.load(path, map_location="cpu")
            state_dict = ckpt.get("model_state_dict", ckpt)
            model.load_state_dict(remap_state_dict(state_dict), strict=False)
            model.to(self.device)
            model.eval()
            self.backbones[name] = model
        print(f"[Vision] Loaded {len(self.backbones)} backbones on {self.device}")

        # FAISS — optional, fails gracefully
        try:
            self.faiss = FAISSRetriever(BANK_FEATURES, BANK_LABELS, k=KNN_K)
        except Exception as e:
            print(f"[Vision] FAISS not loaded (non-fatal): {e}")
            self.faiss = None

        # FAISS state from last gate() call — for UI display only
        self._last_used_faiss: bool = False
        self._last_faiss_similarity: Optional[float] = None

        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def predict(
        self, image: Image.Image
    ) -> Tuple[List[Tuple[str, float]], torch.Tensor]:
        """
        Run ensemble inference.
        Returns (top5_predictions, img_tensor).
        top5_predictions: list of (class_name, confidence_pct).
        img_tensor: pass directly to gate() — no side-channel needed.
        """
        img_tensor = self.transform(image).unsqueeze(0).to(self.device)

        all_probs = []
        for model in self.backbones.values():
            with torch.no_grad():
                probs = torch.softmax(model(img_tensor), dim=1)
            all_probs.append(probs)

        avg_probs = torch.stack(all_probs).mean(dim=0)
        probs_np = avg_probs.cpu().numpy()[0]
        top_indices = probs_np.argsort()[::-1][:5]
        predictions = [(self.class_names[i], probs_np[i] * 100) for i in top_indices]
        return predictions, img_tensor

    def gate(
        self,
        predictions: List[Tuple[str, float]],
        img_tensor: Optional[torch.Tensor] = None,
    ) -> Tuple[Optional[str], bool, Optional[float]]:
        """
        Returns (class_name, used_faiss, faiss_similarity).

        class_name       — class to use for RAG/advisory, or None to abstain.
        used_faiss       — True if FAISS k-NN fallback was triggered.
        faiss_similarity — cosine similarity from FAISS query, or None.

        High confidence (>= CONFIDENCE_GATE): returns top ensemble prediction.
        Low confidence: tries FAISS k-NN fallback.
        FAISS similarity below FAISS_MIN_SIMILARITY: abstains (class_name=None).

        Instance attributes _last_used_faiss / _last_faiss_similarity are still
        updated for any legacy code that reads them, but callers should prefer
        unpacking the returned tuple.
        """
        self._last_used_faiss = False
        self._last_faiss_similarity = None

        if predictions and predictions[0][1] >= CONFIDENCE_GATE:
            return predictions[0][0], False, None

        if self.faiss is not None and img_tensor is not None:
            feats = []
            for backbone in self.backbones.values():
                with torch.no_grad():
                    f = backbone.extract_features(img_tensor).cpu().numpy()
                feats.append(f)
            combined = np.concatenate(feats, axis=1)

            class_idx, similarity = self.faiss.query(combined)

            if similarity < FAISS_MIN_SIMILARITY:
                print(f"[Vision] FAISS similarity {similarity:.3f} below "
                      f"threshold {FAISS_MIN_SIMILARITY} — abstaining")
                self._last_used_faiss = False
                self._last_faiss_similarity = similarity
                return None, False, similarity

            class_name = self.class_names[class_idx]
            self._last_used_faiss = True
            self._last_faiss_similarity = similarity
            print(f"[Vision] FAISS fallback → {class_name} (sim={similarity:.3f})")
            return class_name, True, similarity

        return None, False, None

Overwriting /kaggle/working/cropquery/vision.py


### 3c · knowledge_base.py

**Fix vs old version:** Kisan chunks truncated at sentence boundary (`rsplit`) not mid-word.

In [6]:
%%writefile /kaggle/working/cropquery/knowledge_base.py
"""
CropQuery — Knowledge Base Builder
"""
import os, re, json
from typing import List, Dict, Any, Optional

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

from .config import CHROMA_DIR, COLLECTION_NAME, EMBED_MODEL, HF_KISAN_DATASET, HF_AGROBENCH_DATASET

_KISAN_QUESTION_COLS = ["question","questions","Question","Questions","query","Query","farmer_query","input","text"]
_KISAN_ANSWER_COLS   = ["answer","answers","Answer","Answers","expert_answer","response","Response","output","label"]

def _get_col(row: dict, candidates: list) -> str:
    for key in candidates:
        val = row.get(key)
        if val and str(val).strip():
            return str(val).strip()
    return ""

def _parse_kisan_dataset() -> List[Dict[str, Any]]:
    try:
        from datasets import load_dataset
    except ImportError:
        print("[KB] datasets not installed — skipping Kisan source")
        return []
    print("[KB] Loading Kisan Call Centre dataset...")
    try:
        ds = load_dataset(HF_KISAN_DATASET, split="train")
    except Exception as e:
        print(f"[KB] Could not load Kisan dataset: {e}")
        return []
    if len(ds) == 0:
        return []
    print(f"[KB] Kisan columns: {list(ds[0].keys())}")
    chunks, skipped = [], 0
    for i, row in enumerate(ds):
        question = _get_col(row, _KISAN_QUESTION_COLS)
        answer   = _get_col(row, _KISAN_ANSWER_COLS)
        if not question or not answer:
            skipped += 1
            continue
        text = f"Farmer Query: {question}\nExpert Answer: {answer}"
        # FIXED: truncate at sentence boundary, not mid-word
        if len(text) > 1500:
            text = text[:1500].rsplit(". ", 1)[0] + "."
        chunks.append({
            "id":       f"kisan_{i}",
            "text":     text,
            "metadata": {
                "source":     "kisan_call_centre",
                "crop":       "general",
                "condition":  "general",
                "chunk_type": "qa_pair",
                "language":   "en",
            },
        })
    if skipped:
        print(f"[KB] Kisan: skipped {skipped} rows with missing question/answer")
    print(f"[KB] Kisan: {len(chunks)} Q&A pairs loaded")
    return chunks

def _build_agrobench_chunks() -> List[Dict[str, Any]]:
    try:
        from datasets import load_dataset
        import os
        token = os.environ.get("HF_TOKEN")
        ds = load_dataset(
            HF_AGROBENCH_DATASET,
            split="train",
            token=token,
        )
        chunks = []
        for i, row in enumerate(ds):
            chunks.append({
                "id":       f"agrobench_{i}",
                "text":     row["text"],
                "metadata": {
                    "source":     "agrobench",
                    "crop":       row["crop"],
                    "condition":  row["condition"],
                    "chunk_type": "disease_entry",
                    "language":   "en",
                },
            })
        print(f"[KB] AgroBench: {len(chunks)} entries loaded from HuggingFace")
        return chunks
    except Exception as e:
        print(f"[KB] AgroBench HF load failed: {e}")
        return []

class KnowledgeBaseBuilder:
    def __init__(self, chroma_dir: str = CHROMA_DIR, embed_model: str = EMBED_MODEL):
        os.makedirs(chroma_dir, exist_ok=True)
        self.client = chromadb.PersistentClient(path=chroma_dir)
        self.ef = SentenceTransformerEmbeddingFunction(model_name=embed_model)
        self.collection = self.client.get_or_create_collection(
            name=COLLECTION_NAME,
            embedding_function=self.ef,
            metadata={"hnsw:space": "cosine"},
        )

    def build(self, force_rebuild: bool = False, batch_size: int = 64) -> int:
        if force_rebuild:
            print("[KB] Rebuilding from scratch...")
            self.client.delete_collection(COLLECTION_NAME)
            self.collection = self.client.get_or_create_collection(
                name=COLLECTION_NAME,
                embedding_function=self.ef,
                metadata={"hnsw:space": "cosine"},
            )
        elif self.collection.count() > 0:
            print(f"[KB] Collection already has {self.collection.count()} docs. "
                  "Pass force_rebuild=True to overwrite.")
            return self.collection.count()

        all_chunks = _build_agrobench_chunks() + _parse_kisan_dataset()
        if not all_chunks:
            print("[KB] No chunks to ingest!")
            return 0

        seen, unique = set(), []
        for c in all_chunks:
            if c["id"] not in seen:
                seen.add(c["id"])
                unique.append(c)

        total = len(unique)
        print(f"[KB] Upserting {total} chunks into ChromaDB...")
        for start in range(0, total, batch_size):
            batch = unique[start:start + batch_size]
            self.collection.upsert(
                ids=[c["id"] for c in batch],
                documents=[c["text"] for c in batch],
                metadatas=[c["metadata"] for c in batch],
            )
        print(f"[KB] Done. Collection size: {self.collection.count()} documents")
        return self.collection.count()

    def stats(self) -> Dict[str, Any]:
        count = self.collection.count()
        sources: Dict[str, int] = {}
        sample = self.collection.get(limit=min(count, 1000), include=["metadatas"])
        for meta in sample["metadatas"]:
            src = meta.get("source", "unknown")
            sources[src] = sources.get(src, 0) + 1
        return {"total_documents": count, "source_breakdown": sources}


Overwriting /kaggle/working/cropquery/knowledge_base.py


### 3d · retriever.py

**Fix vs old version:** Removed unused `MIN_CROP_CHUNKS` import.

In [7]:
%%writefile /kaggle/working/cropquery/retriever.py
"""
CropQuery — RAG Retriever
"""
import re
from typing import List, Dict, Any, Optional

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

from .config import (
    CHROMA_DIR, COLLECTION_NAME, EMBED_MODEL,
    TOP_CHUNKS, TOP_K_AGROBENCH, TOP_K_KISAN,
)

_RAG_JARGON_MAP = {
    "new flush":          "new leaves",
    "flush emergence":    "new leaves appearing",
    "flush":              "new leaves",
    "lesions":            "spots",
    "necrosis":           "dead patches",
    "necrotic":           "dead",
    "inoculum":           "disease source",
    "sporulation":        "disease spreading",
    "oviposition":        "egg-laying",
    "larval mining":      "pest damage inside leaves",
    "foliar application": "leaf spray",
    "phytotoxicity":      "plant damage from chemicals",
    "chlorosis":          "yellowing",
    "defoliation":        "leaf fall",
    "stippling":          "tiny spots",
    "instar":             "pest stage",
}

def _clean_rag_chunk(text: str) -> str:
    for term, replacement in _RAG_JARGON_MAP.items():
        text = re.sub(re.escape(term), replacement, text, flags=re.IGNORECASE)
    return text


class CropRetriever:

    def __init__(
        self,
        chroma_dir: str = CHROMA_DIR,
        embed_model: str = EMBED_MODEL,
        top_k: int = TOP_CHUNKS,
    ):
        self.top_k = top_k
        client = chromadb.PersistentClient(path=chroma_dir)
        ef = SentenceTransformerEmbeddingFunction(model_name=embed_model)
        self.collection = client.get_collection(
            name=COLLECTION_NAME,
            embedding_function=ef,
        )
        print(f"[RAG] Connected to ChromaDB. {self.collection.count()} documents.")

    def retrieve(
        self,
        predicted_class: Optional[str],
        farmer_question: str = "What should I do?",
    ) -> List[Dict[str, Any]]:
        query     = self._build_query(predicted_class, farmer_question)
        crop      = self._extract_crop(predicted_class)
        condition = self._extract_condition(predicted_class)

        crop_chunks  = []
        kisan_chunks = []

        # 1. Crop + condition specific agrobench chunks
        if crop:
            try:
                where_filter = {"$and": [
                    {"source": {"$eq": "agrobench"}},
                    {"crop":   {"$eq": crop}},
                ]}
                if condition:
                    where_filter["$and"].append({"condition": {"$eq": condition}})
                r = self.collection.query(
                    query_texts=[query],
                    n_results=TOP_K_AGROBENCH,
                    where=where_filter,
                    include=["documents", "metadatas", "distances"],
                )
                crop_chunks = self._parse_results(r)

                # If condition filter returned nothing, retry crop-only
                if not crop_chunks and condition:
                    r = self.collection.query(
                        query_texts=[query],
                        n_results=TOP_K_AGROBENCH,
                        where={"$and": [
                            {"source": {"$eq": "agrobench"}},
                            {"crop":   {"$eq": crop}},
                        ]},
                        include=["documents", "metadatas", "distances"],
                    )
                    crop_chunks = self._parse_results(r)
            except Exception as e:
                print(f"[RAG] Crop query failed: {e}")

        # 2. Kisan Q&A chunks
        try:
            r = self.collection.query(
                query_texts=[query],
                n_results=TOP_K_KISAN,
                where={"source": {"$eq": "kisan_call_centre"}},
                include=["documents", "metadatas", "distances"],
            )
            kisan_chunks = self._parse_results(r)
        except Exception as e:
            print(f"[RAG] Kisan query failed: {e}")

        # 3. Fallback — unfiltered if still nothing crop-specific
        if not crop_chunks:
            try:
                r = self.collection.query(
                    query_texts=[query],
                    n_results=TOP_K_AGROBENCH,
                    include=["documents", "metadatas", "distances"],
                )
                crop_chunks = self._parse_results(r)
                print("[RAG] Warning: using unfiltered fallback — verify chunks are relevant")
            except Exception as e:
                print(f"[RAG] Unfiltered fallback failed: {e}")

        # 4. Post-filter by condition to reduce mixed chunks
        if condition and crop_chunks:
            condition_lower = condition.lower()
            filtered = [
                c for c in crop_chunks
                if c["metadata"].get("source") != "agrobench"
                or c["metadata"].get("condition", "").lower() == condition_lower
                or c["metadata"].get("condition", "").lower() in ("general", "unknown", "")
            ]
            if filtered:
                crop_chunks = filtered

        return (crop_chunks + kisan_chunks)[:(TOP_K_AGROBENCH + TOP_K_KISAN)]

    def build_rag_context(self, chunks: List[Dict[str, Any]]) -> str:
        if not chunks:
            return "No specific knowledge retrieved. Use general agronomy principles."
        MAX_CHUNK_CHARS = 400
        lines = ["=== Retrieved Agricultural Knowledge ==="]
        for i, chunk in enumerate(chunks, 1):
            src  = chunk["metadata"].get("source", "unknown")
            crop = chunk["metadata"].get("crop", "")
            cond = chunk["metadata"].get("condition", "")
            label = f"[{i}] Source: {src}"
            if crop and crop not in ("Unknown", "general"):
                label += f" | Crop: {crop}"
            if cond and cond != "general":
                label += f" | Disease: {cond}"
            lines.append(label)
            text = _clean_rag_chunk(chunk["text"])
            if len(text) > MAX_CHUNK_CHARS:
                text = text[:MAX_CHUNK_CHARS].rsplit(". ", 1)[0] + "."
            lines.append(text)
            lines.append("")
        return "\n".join(lines)

    @staticmethod
    def _extract_crop(predicted_class: Optional[str]) -> Optional[str]:
        if not predicted_class:
            return None
        return predicted_class.split("__")[0].replace("_", " ")

    @staticmethod
    def _extract_condition(predicted_class: Optional[str]) -> Optional[str]:
        if not predicted_class or "__" not in predicted_class:
            return None
        return predicted_class.split("__", 1)[1].replace("_", " ").title()

    @staticmethod
    def _build_query(predicted_class: Optional[str], farmer_question: str) -> str:
        parts = []
        if predicted_class:
            crop, _, cond = predicted_class.partition("__")
            parts.append(f"{crop.replace('_', ' ')} {cond.replace('_', ' ')} disease symptoms treatment")
        parts.append(farmer_question)
        return " ".join(parts)

    @staticmethod
    def _parse_results(results: Dict[str, Any]) -> List[Dict[str, Any]]:
        chunks = []
        if not results or not results.get("documents"):
            return chunks
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            chunks.append({"text": doc, "metadata": meta, "distance": dist})
        return chunks


Overwriting /kaggle/working/cropquery/retriever.py


### 3e · advisor.py

In [8]:
%%writefile /kaggle/working/cropquery/advisor.py
"""
CropQuery — LLaVA Advisor

Dual-stream inference:
  Stream A  →  3-backbone ensemble (cuda:0)  →  class label + confidence
  Stream B  →  LLaVA vision encoder (cuda:1) →  visual symptom description

Stream B runs in two stages:
  Stage 1: short visual description prompt  (~80 tokens) — fast
  Stage 2: full advisory generation prompt  (~768 tokens)

Stage 1 output feeds into Stage 2 prompt as `visual_symptoms`,
replacing the old text-only "Predicted class: ..." placeholder.
This is what justifies using LLaVA instead of plain LLaMA.
"""
import re
from typing import Optional, Generator
from PIL import Image
from transformers import TextIteratorStreamer
from threading import Thread

from .config import (
    LLAVA_MODEL_ID, LLAVA_MAX_TOKENS, LOAD_IN_4BIT,
    CONFIDENCE_GATE, LLAVA_DEVICE,
)


# ── Spray timing helper ────────────────────────────────────────────────────────

def _spray_timing_warning(weather_ctx: dict) -> str:
    if not weather_ctx:
        return ""
    from .context_fetcher import spray_timing_advisory
    return spray_timing_advisory(weather_ctx)


# ── Jargon simplifier ─────────────────────────────────────────────────────────

_JARGON_MAP = {
    "new flush":           "new leaves",
    "flush emergence":     "new leaves appearing",
    "flush":               "new leaves",
    "inoculum":            "disease source",
    "sporulation":         "disease spreading",
    "lesions":             "spots",
    "necrosis":            "dead patches",
    "necrotic":            "dead",
    "vector":              "carrier insect",
    "infestation":         "attack",
    "larval mining":       "pest damage inside leaves",
    "oviposition":         "egg-laying",
    "instar":              "stage of pest",
    "systemic":            "absorbed by the plant",
    "contact insecticide": "spray that kills on contact",
    "foliar application":  "leaf spray",
    "phytotoxicity":       "plant damage from chemicals",
    "stippling":           "tiny spots",
    "chlorosis":           "yellowing",
    "defoliation":         "leaf fall",
}


def _clean_jargon(text: str) -> str:
    for jargon, simple in _JARGON_MAP.items():
        text = re.sub(re.escape(jargon), simple, text, flags=re.IGNORECASE)
    return text


# ── Safety validator ───────────────────────────────────────────────────────────

_FUNGICIDE_KEYWORDS = {
    "carbendazim", "mancozeb", "copper oxychloride", "chlorothalonil",
    "metalaxyl", "propiconazole", "tebuconazole", "captan", "myclobutanil",
    "trifloxystrobin", "azoxystrobin", "hexaconazole", "bordeaux",
    "copper hydroxide", "fosetyl", "cymoxanil",
}

_PEST_KEYWORDS = {
    "leaf miner", "spider mite", "aphid", "whitefly", "mealybug",
    "thrips", "scale", "weevil", "borer", "caterpillar", "locust",
    "fruit fly", "jassid", "leafhopper",
}


def _validate_advice(advice: str, predicted_class: Optional[str]) -> str:
    if not predicted_class:
        return advice
    condition = predicted_class.split("__", 1)[-1].replace("_", " ").lower()
    is_pest = any(kw in condition for kw in _PEST_KEYWORDS)
    if not is_pest:
        return advice
    advice_lower = advice.lower()
    flagged = [kw for kw in _FUNGICIDE_KEYWORDS if kw in advice_lower]
    if flagged:
        warning = (
            "\n\n⚠️ [Safety Check] This appears to be a PEST problem, not a fungal disease. "
            f"The following chemicals listed above are fungicides and will NOT work against this pest: "
            f"{', '.join(flagged).title()}. "
            "Please consult your local agriculture officer for the correct insecticide."
        )
        print(f"[Validator] PEST/FUNGICIDE MISMATCH detected: {flagged}")
        return advice + warning
    return advice


# ── Non-leaf detection ────────────────────────────────────────────────────────

# Phrases LLaVA uses when there is no plant/leaf visible.
# Checked in the Stage 1 visual description output.
_NO_LEAF_PHRASES = [
    "no plant", "no leaf", "not a leaf", "no crop", "no vegetation",
    "cannot identify", "not a plant", "no disease visible",
    "no agricultural", "unclear image", "no leaves",
]

_NO_LEAF_RESPONSE = """❓ आपका जवाब (Your Answer)
* No plant or leaf was detected in this image.
* Please upload a clear photo of a leaf or crop to get disease advice.

🌿 समस्या (Problem)
* Image does not show a plant leaf.
* Cannot diagnose without a clear leaf photo.

✅ क्या करें (What to do now)
* Take a close-up photo of the affected leaf in natural daylight.
* Make sure the leaf fills most of the frame.
* Avoid blurry or very dark photos.

⚠️ सुरक्षा (Safety Notice)
* Always confirm chemical use with your local agriculture officer before applying."""


def _is_no_leaf(visual_description: str) -> bool:
    desc_lower = visual_description.lower()
    return any(phrase in desc_lower for phrase in _NO_LEAF_PHRASES)


# ── Visual description prompt (Stage 1) ───────────────────────────────────────

_VISUAL_DESCRIPTION_PROMPT = """Look at this image carefully.

If you can see a plant leaf or crop:
Describe ONLY what you see on the leaf in 2-3 short bullet points:
- Colour of affected areas (yellow, brown, black, white spots?)
- Location on leaf (edges, centre, whole leaf, underside?)
- Spread (small patches, large areas, covering most of leaf?)
- Any visible pests, webbing, or powdery coating?

If there is NO plant or leaf visible in this image, reply with exactly:
"No plant or leaf visible in this image."

Keep your answer under 60 words. Do not diagnose. Do not recommend treatment. Just describe."""


# ── Main advisory prompt (Stage 2) ────────────────────────────────────────────

_SYSTEM_PROMPT = """You are CropQuery — a trusted, practical agricultural advisor for Indian farmers.
You speak like a helpful village extension worker: calm, honest, simple, and caring.

========================
YOUR INPUTS
========================
Visual Symptoms (what the camera sees): {visual_symptoms}
Knowledge from database: {retrieved_context}
Farmer's Question: {user_query}

========================
STRICT RULES — FOLLOW EXACTLY
========================
1. RAG ONLY: Recommend ONLY chemicals and treatments found in the knowledge database.
   Never invent pesticide names, dosages, or schedules.
2. NO JARGON: Use only simple words a farmer with no schooling can understand.
   BANNED WORDS: "flush"→"new leaves", "lesions"→"spots", "necrosis"→"dead patches",
   "inoculum"→"disease source", "sporulation"→"disease spreading",
   "vector"→"carrier insect", "foliar application"→"leaf spray",
   "systemic"→"absorbed into the plant".
3. ANSWER THE QUESTION: Answer the farmer's specific question first and directly.
4. FERTILIZER RULE: If asked about fertilizer, explain it does not cure this problem.
5. CURABLE RULE: If asked whether curable, answer YES or NO clearly on the first line.
6. NO GUARANTEES: Never say "definitely", "100% cure", "certainly cured".
7. SHORT LINES: Maximum 10 words per bullet point. No paragraphs.
8. UNCERTAINTY: If symptoms and database do not match well, say so honestly.

========================
OUTPUT FORMAT — COPY EXACTLY
========================

❓ आपका जवाब (Your Answer)
* [Direct answer to the farmer's question in 1-2 simple sentences.]

🌿 समस्या (Problem)
* [Disease or pest name — simple name only]
* [Confidence: High / Medium / Low]
* [What you see on the leaf that tells you this — use the visual symptoms above]

⚠️ स्थिति (Severity)
* [Mild / Moderate / Severe]
* [Why this severity, what damage it is doing]

✅ क्या करें (What to do now)
* Organic: [one simple organic action]
* Chemical: [chemical name + dosage from database only]
* Prevent: [one simple preventive action]

📅 अगले कुछ दिन (Next 7 days)
* Check: [what to look for]
* Act if: [sign that problem is getting worse]

🌦️ मौसम की सलाह (Weather Advice)
* [One rule about when NOT to spray]

📞 कब सलाह लें (When to seek help)
* Call your agriculture officer if: (1) more than 30% of leaves show symptoms, OR (2) symptoms spread to new leaves after 3 days of treatment, OR (3) you are unsure which chemical to use.

⚠️ सुरक्षा (Safety Notice)
* Always confirm chemical use with your local agriculture officer before applying.
* Keep children and animals away from sprayed areas for 48 hours.
* Wear gloves and mask when mixing chemicals.

========================
OUTPUT LANGUAGE
========================
{output_language_instruction}
"""


def _build_prompt(
    rag_context: str,
    farmer_question: str,
    resnet_hint: Optional[str],
    confidence: Optional[float],
    visual_description: str,                   # ← NEW: from Stage 1
    weather_ctx: Optional[dict] = None,
    location_ctx: Optional[dict] = None,
) -> str:
    parts = []

    if resnet_hint and confidence and confidence >= CONFIDENCE_GATE:
        crop, condition = (
            resnet_hint.split("__", 1) if "__" in resnet_hint
            else (resnet_hint, "Unknown")
        )
        crop      = crop.replace("_", " ").title()
        condition = condition.replace("_", " ").title()
        parts.append(
            f"CONFIRMED DIAGNOSIS: Vision model identified this as "
            f"'{condition}' in '{crop}' with {confidence:.1f}% confidence.\n"
            f"Your ENTIRE response must be about '{condition}' in '{crop}'. "
            f"Do NOT discuss any other disease or pest."
        )
        parts.append("")

    if location_ctx:
        loc_parts = []
        for key, label in [
            ("state", "State"), ("district", "District"),
            ("agro_climatic_zone", "Zone"), ("current_season", "Season"),
        ]:
            val = location_ctx.get(key)
            if val:
                loc_parts.append(f"{label}: {val}")
        if loc_parts:
            parts.append("=== Farmer Location ===")
            parts.append(" | ".join(loc_parts))
            parts.append("")

    if weather_ctx:
        w = weather_ctx
        weather_lines = ["=== Current Weather ==="]
        if w.get("temp_c") is not None:
            weather_lines.append(
                f"Temp: {w['temp_c']}°C  |  Humidity: {w.get('humidity_pct', 'N/A')}%"
            )
        if w.get("wind_kmh") is not None:
            weather_lines.append(
                f"Wind: {w['wind_kmh']} km/h  |  "
                f"Rain last 14 days: {w.get('rainfall_14d_mm', 'N/A')} mm"
            )
        if w.get("forecast_7d"):
            forecast_str = ", ".join(
                f"{d['date']}: {d.get('rain_mm', 0):.0f}mm"
                for d in w["forecast_7d"][:3]
            )
            weather_lines.append(f"3-day rain forecast: {forecast_str}")
        parts.append("\n".join(weather_lines))
        parts.append("")
        spray_warn = _spray_timing_warning(weather_ctx)
        if spray_warn:
            parts.append("=== Spray Timing Warning ===")
            parts.append(spray_warn)
            parts.append("")

    q = farmer_question.strip() if farmer_question else "What should I do about this?"
    parts.append(f"=== Farmer's Question ===\n{q}")
    parts.append("")
    parts.append(
        "IMPORTANT REMINDER: Answer the farmer's question FIRST and DIRECTLY. "
        "If they asked about fertilizer, explain it won't cure this. "
        "If they asked if it is curable, say YES or NO clearly."
    )

    context_block = "\n".join(parts)

    return _SYSTEM_PROMPT.format(
        visual_symptoms=visual_description,        # ← now real vision output, not just class name
        retrieved_context=rag_context,
        user_query=q,
        output_language_instruction=(
            "Respond entirely in English. "
            "Use the exact emoji section headers shown above. "
            "Do not write paragraphs — use short bullet points only."
        ),
    ) + "\n\n" + context_block


# ── LLaVAAdvisor ──────────────────────────────────────────────────────────────

class LLaVAAdvisor:
    """
    Wraps LLaVA-Next-Mistral-7B for dual-stream image-grounded agricultural advice.

    Stage 1: _describe_image() — short visual description (~80 tokens).
             Checks for non-leaf images and short-circuits if detected.
    Stage 2: generate() / generate_streaming() — full advisory generation.
             visual_description from Stage 1 is injected as {visual_symptoms}
             in the system prompt, so LLaVA grounds its advice in what it sees.
    """

    def __init__(
        self,
        model_id: str = LLAVA_MODEL_ID,
        load_in_4bit: bool = LOAD_IN_4BIT,
        device_map: str = LLAVA_DEVICE,
    ):
        import torch, gc
        from transformers import (
            LlavaNextProcessor,
            LlavaNextForConditionalGeneration,
            BitsAndBytesConfig,
        )
    
        # ── Flush GPU 1 before loading ─────────────────────────────────
        gc.collect()
        torch.cuda.empty_cache()
        free_gb = (torch.cuda.get_device_properties(1).total_memory
                   - torch.cuda.memory_allocated(1)) / 1e9
        print(f"[LLaVA] GPU 1 free before load: {free_gb:.2f} GiB")
        if free_gb < 9.0:
            raise RuntimeError(
                f"[LLaVA] GPU 1 only has {free_gb:.1f} GiB free — need ~9 GiB for 4-bit LLaVA. "
                "Clear GPU 1 memory before loading."
            )
    
        print(f"[LLaVA] Loading {model_id} ({'4-bit' if load_in_4bit else 'full'})...")
        quant_cfg = None
        if load_in_4bit:
            quant_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
        self.processor = LlavaNextProcessor.from_pretrained(model_id)
        self.model = LlavaNextForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=quant_cfg,
            device_map={"": 1},          # ← explicit: entire model on GPU 1
            torch_dtype=torch.float16 if not load_in_4bit else None,
            low_cpu_mem_usage=True,
        )
        self.model.eval()
        print("[LLaVA] Model loaded.")

    def _prepare_inputs(self, image: Image.Image, prompt: str):
        if image.mode != "RGB":
            image = image.convert("RGB")
        conversation = [{
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": prompt}],
        }]
        prompt_text = self.processor.apply_chat_template(
            conversation, add_generation_prompt=True
        )
        return self.processor(
            images=image, text=prompt_text, return_tensors="pt"
        ).to(self.model.device)

    def _describe_image(self, image: Image.Image) -> str:
        """
        Stage 1: Run a short visual description pass.
        Returns a 2-3 bullet description of what the model sees on the leaf,
        or a no-leaf sentinel string if no plant is visible.
        ~80 tokens — fast.
        """
        import torch
        inputs = self._prepare_inputs(image, _VISUAL_DESCRIPTION_PROMPT)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,           # greedy — deterministic description
            )
        input_len = inputs["input_ids"].shape[1]
        description = self.processor.decode(
            output_ids[0][input_len:], skip_special_tokens=True
        ).strip()
        print(f"[LLaVA] Visual description: {description[:120]}...")
        return description

    def generate(
        self,
        image: Image.Image,
        rag_context: str,
        farmer_question: str = "What should I do?",
        resnet_hint: Optional[str] = None,
        confidence: Optional[float] = None,
        max_new_tokens: int = LLAVA_MAX_TOKENS,
        weather_ctx: Optional[dict] = None,
        location_ctx: Optional[dict] = None,
    ) -> str:
        import torch

        # Stage 1: get visual description (or detect non-leaf)
        visual_description = self._describe_image(image)
        if _is_no_leaf(visual_description):
            print("[LLaVA] Non-leaf image detected — skipping advisory generation.")
            return _NO_LEAF_RESPONSE

        prompt = _build_prompt(
            rag_context, farmer_question, resnet_hint,
            confidence, visual_description, weather_ctx, location_ctx,
        )
        inputs = self._prepare_inputs(image, prompt)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.3,
                repetition_penalty=1.15,
            )
        input_len = inputs["input_ids"].shape[1]
        raw = self.processor.decode(
            output_ids[0][input_len:], skip_special_tokens=True
        ).strip()
        return _validate_advice(_clean_jargon(raw), resnet_hint)

    def generate_streaming(
        self,
        image: Image.Image,
        rag_context: str,
        farmer_question: str = "What should I do?",
        resnet_hint: Optional[str] = None,
        confidence: Optional[float] = None,
        weather_ctx: Optional[dict] = None,
        location_ctx: Optional[dict] = None,
    ) -> TextIteratorStreamer:
        # Stage 1: visual description (non-streaming — it's short)
        visual_description = self._describe_image(image)
        if _is_no_leaf(visual_description):
            print("[LLaVA] Non-leaf image detected — returning static response.")
            # Wrap the static response in a trivial streamer-compatible iterator
            class _StaticStreamer:
                def __iter__(self):
                    yield _NO_LEAF_RESPONSE
            return _StaticStreamer()

        prompt = _build_prompt(
            rag_context, farmer_question, resnet_hint,
            confidence, visual_description, weather_ctx, location_ctx,
        )
        inputs = self._prepare_inputs(image, prompt)
        streamer = TextIteratorStreamer(
            self.processor.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )
        Thread(
            target=self.model.generate,
            kwargs=dict(
                **inputs,
                streamer=streamer,
                max_new_tokens=LLAVA_MAX_TOKENS,
                do_sample=True,
                temperature=0.3,
                repetition_penalty=1.15,
            ),
        ).start()
        return streamer

Overwriting /kaggle/working/cropquery/advisor.py


### 3f · translator.py

In [9]:
%%writefile /kaggle/working/cropquery/translator.py
"""
CropQuery — Multilingual Translation (deep-translator backend)
NOTE: Uses GoogleTranslator from deep-translator, NOT IndicTrans2.
"""
from typing import Optional

_GOOGLE_LANG_CODES = {
    "Hindi":     "hi",
    "Tamil":     "ta",
    "Telugu":    "te",
    "Kannada":   "kn",
    "Malayalam": "ml",
    "Bengali":   "bn",
    "Marathi":   "mr",
    "Gujarati":  "gu",
    "Punjabi":   "pa",
    "Odia":      "or",
    "Urdu":      "ur",
}


class IndicTranslator:
    """
    Wraps deep-translator (GoogleTranslator) for English → Indian language
    translation. Same public interface as the original IndicTrans2 class.
    """

    def __init__(self, model_id: str = None):
        self._loaded: Optional[bool] = None
        self._translator_cls = None

    def _load(self):
        if self._loaded is not None:
            return
        try:
            from deep_translator import GoogleTranslator
            self._translator_cls = GoogleTranslator
            self._loaded = True
            print("[Trans] deep-translator (GoogleTranslator) ready.")
        except ImportError:
            print("[Trans] WARNING: deep-translator not installed. pip install deep-translator")
            self._loaded = False

    def translate(
        self,
        text: str,
        target_language: str = "Hindi",
        src_lang: str = "eng_Latn",
    ) -> str:
        google_code = _GOOGLE_LANG_CODES.get(target_language)
        if google_code is None:
            return text  # English — no translation needed
        self._load()
        if self._loaded is False:
            return text
        try:
            if len(text) <= 4500:
                return self._translator_cls(source="en", target=google_code).translate(text) or text
            else:
                return self._translate_chunked(text, google_code)
        except Exception as e:
            print(f"[Trans] Translation failed: {e} — returning English")
            return text

    def _translate_chunked(self, text: str, google_code: str) -> str:
        chunks = text.split("\n\n")
        translated = []
        t = self._translator_cls(source="en", target=google_code)
        for chunk in chunks:
            try:
                translated.append(t.translate(chunk) if chunk.strip() else chunk)
            except Exception:
                translated.append(chunk)
        return "\n\n".join(translated)

    @staticmethod
    def available_languages() -> list:
        return ["English"] + list(_GOOGLE_LANG_CODES.keys())

    @staticmethod
    def language_code(display_name: str) -> Optional[str]:
        return _GOOGLE_LANG_CODES.get(display_name)


Overwriting /kaggle/working/cropquery/translator.py


### 3g · context_fetcher.py

In [10]:
%%writefile /kaggle/working/cropquery/context_fetcher.py
"""
CropQuery — Context Fetcher
B1: Weather (Open-Meteo, free, no key)
B2: Location + agro-climatic zone (IP-based)
D1: Irrigation advisory
"""
import datetime
from typing import Optional, Dict, Any

try:
    import requests
    _REQUESTS_OK = True
except ImportError:
    _REQUESTS_OK = False

from .config import WEATHER_API_URL

_STATE_ZONE = {
    "Punjab":           "Trans-Gangetic Plains",
    "Haryana":          "Trans-Gangetic Plains",
    "Uttar Pradesh":    "Upper Gangetic Plains",
    "Bihar":            "Middle Gangetic Plains",
    "West Bengal":      "Lower Gangetic Plains",
    "Maharashtra":      "Western Plateau and Hills",
    "Karnataka":        "Southern Plateau and Hills",
    "Tamil Nadu":       "East Coast Plains and Hills",
    "Andhra Pradesh":   "East Coast Plains and Hills",
    "Telangana":        "Southern Plateau and Hills",
    "Rajasthan":        "Western Dry Region",
    "Gujarat":          "Gujarat Plains and Hills",
    "Madhya Pradesh":   "Central Plateau and Hills",
    "Odisha":           "East Coast Plains and Hills",
    "Kerala":           "Western Ghats and Coastal Plains",
    "Assam":            "North-Eastern Hills",
    "Himachal Pradesh": "Western Himalayas",
    "Uttarakhand":      "Western Himalayas",
    "Jharkhand":        "Eastern Plateau and Hills",
    "Chhattisgarh":     "Eastern Plateau and Hills",
}

def _current_season() -> str:
    m = datetime.datetime.now().month
    if 6 <= m <= 10: return "Kharif"
    elif m >= 11 or m <= 3: return "Rabi"
    return "Zaid"

_CROP_WATER_REQ: Dict[str, Dict[str, float]] = {
    "Tomato":  {"seedling":15,"vegetative":25,"flowering":40,"fruiting":45,"default":35},
    "Potato":  {"seedling":20,"vegetative":35,"flowering":40,"fruiting":40,"default":35},
    "Wheat":   {"seedling":18,"vegetative":28,"flowering":35,"fruiting":30,"default":28},
    "Rice":    {"seedling":40,"vegetative":55,"flowering":60,"fruiting":50,"default":50},
    "Corn":    {"seedling":20,"vegetative":30,"flowering":45,"fruiting":40,"default":35},
    "Grape":   {"seedling":25,"vegetative":35,"flowering":30,"fruiting":40,"default":33},
    "Apple":   {"seedling":20,"vegetative":30,"flowering":35,"fruiting":40,"default":30},
    "Cashew":  {"seedling":15,"vegetative":20,"flowering":25,"fruiting":30,"default":22},
    "Pea":     {"seedling":15,"vegetative":22,"flowering":30,"fruiting":28,"default":24},
    "default": {"default":30},
}

_SATURATION_THRESHOLD_MM = 100
_CARRYOVER_FACTOR        = 0.30


def spray_timing_advisory(weather_ctx: dict) -> str:
    if not weather_ctx:
        return "Check local weather before spraying."
    wind      = weather_ctx.get("wind_kmh", 0) or 0
    rain_prob = weather_ctx.get("rain_prob_6h", 0) or 0
    warnings  = []
    if wind > 15:
        warnings.append(
            f"⚠️ Wind is {wind:.0f} km/h — do not spray now. Wait for calm weather (<15 km/h)."
        )
    if rain_prob > 40:
        warnings.append(
            f"⚠️ Rain expected in next 6 hours ({rain_prob:.0f}% chance) — "
            f"spraying now will wash off the medicine. Wait."
        )
    if warnings:
        return "\n".join(warnings)
    return (
        f"✅ Good spraying conditions — wind {wind:.0f} km/h, "
        f"rain probability {rain_prob:.0f}% in next 6 hours. Safe to spray."
    )


class ContextFetcher:
    def __init__(self):
        self._location: Optional[dict] = None

    def _resolve_location(self) -> Optional[dict]:
        if not _REQUESTS_OK:
            return None
        try:
            geo   = requests.get("https://ipapi.co/json/", timeout=5).json()
            lat   = float(geo.get("latitude", 0))
            lon   = float(geo.get("longitude", 0))
            state = geo.get("region", "")
            zone  = _STATE_ZONE.get(state, None)
            return {
                "state":              state,
                "district":           geo.get("city", ""),
                "latitude":           lat,
                "longitude":          lon,
                "agro_climatic_zone": zone,  # None for non-Indian locations
                "current_season":     _current_season(),
            }
        except Exception as e:
            print(f"[ContextFetcher] Location failed: {e}")
            return None

    def get_location(self) -> dict:
        if self._location is None:
            self._location = self._resolve_location() or {}
        return self._location

    def get_weather(self) -> dict:
        if not _REQUESTS_OK:
            return {}
        loc = self.get_location()
        lat = loc.get("latitude") or 28.6
        lon = loc.get("longitude") or 77.2
        try:
            params = {
                "latitude":      lat, "longitude":    lon,
                "current":       "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
                "hourly":        "precipitation_probability",
                "daily":         "precipitation_sum,temperature_2m_max",
                "past_days":     14, "forecast_days": 7, "timezone": "auto",
            }
            data    = requests.get(WEATHER_API_URL, params=params, timeout=8).json()
            current = data.get("current", {})
            daily   = data.get("daily", {})
            hourly  = data.get("hourly", {})
            precip_list  = daily.get("precipitation_sum", [])
            rainfall_14d = sum(p for p in precip_list[:14] if p) if precip_list else None
            rain_probs   = hourly.get("precipitation_probability", [])
            dates, rains, temps = (
                daily.get("time", []), daily.get("precipitation_sum", []),
                daily.get("temperature_2m_max", [])
            )
            forecast_7d = [
                {"date": d, "rain_mm": r or 0, "temp_max_c": t}
                for d, r, t in zip(dates[-7:], rains[-7:], temps[-7:])
            ]
            return {
                "temp_c":          current.get("temperature_2m"),
                "humidity_pct":    current.get("relative_humidity_2m"),
                "wind_kmh":        current.get("wind_speed_10m"),
                "rainfall_14d_mm": round(rainfall_14d, 1) if rainfall_14d is not None else None,
                "rain_prob_6h":    max(rain_probs[:6]) if rain_probs else 0,
                "forecast_7d":     forecast_7d,
            }
        except Exception as e:
            print(f"[ContextFetcher] Weather fetch failed: {e}")
            return {}

    def get_irrigation_advice(
        self,
        crop: Optional[str] = None,
        growth_stage: str = "default",
        weather_ctx: Optional[dict] = None,
    ) -> str:
        w            = weather_ctx or {}
        forecast_7d  = w.get("forecast_7d", [])
        rainfall_14d = w.get("rainfall_14d_mm") or 0
        crop_key     = crop if crop in _CROP_WATER_REQ else "default"
        stage_key    = growth_stage if growth_stage in _CROP_WATER_REQ.get(crop_key, {}) else "default"
        water_need   = _CROP_WATER_REQ[crop_key].get(stage_key, 30)
        if not forecast_7d:
            return (
                f"[Irrigation] {crop or 'Crop'} needs ~{water_need} mm/week. "
                "Weather forecast unavailable — irrigate based on soil moisture."
            )
        rain_next_7d     = sum(d.get("rain_mm", 0) for d in forecast_7d[:7])
        recent_carryover = min(rainfall_14d * _CARRYOVER_FACTOR, water_need)
        effective_supply = rain_next_7d + recent_carryover
        deficit_mm       = max(0, water_need - effective_supply)
        saturation_note  = ""
        if rainfall_14d > _SATURATION_THRESHOLD_MM:
            saturation_note = (
                f" ⚠️ Heavy rain in last 14 days ({rainfall_14d:.0f} mm) — "
                "soil may be saturated. Check before irrigating."
            )
        best_day = sorted(forecast_7d[:4], key=lambda d: d.get("rain_mm", 0))[0]["date"]
        crop_label = crop or "Your crop"
        if deficit_mm <= 0:
            return (
                f"[Irrigation] {crop_label} needs {water_need} mm/week. "
                f"Forecast rain ({rain_next_7d:.0f} mm) + carry-over ({recent_carryover:.0f} mm) "
                f"covers this week — no additional irrigation needed." + saturation_note
            )
        return (
            f"[Irrigation] {crop_label} needs {water_need} mm/week. "
            f"Forecast: {rain_next_7d:.0f} mm + ~{recent_carryover:.0f} mm carry-over "
            f"= {effective_supply:.0f} mm — irrigate {deficit_mm:.0f} mm deficit. "
            f"Best window: {best_day} (lowest rain). "
            f"Avoid irrigating on days with >40% rain probability." + saturation_note
        )


Overwriting /kaggle/working/cropquery/context_fetcher.py


### 3h · pipeline.py

**Fix vs old version:** `pipeline.run()` now unpacks `(preds, tensor)` tuple from `predict()` — no more `_last_tensor` side-channel.

In [11]:
%%writefile /kaggle/working/cropquery/pipeline.py
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from PIL import Image
import re

from .config import CONFIDENCE_GATE, BACKBONE_DEVICE

_CHEMICAL_PRESERVE = [
    "Carbendazim 50 WP","Carbendazim","Mancozeb 75 WP","Mancozeb",
    "Copper Oxychloride","Copper Hydroxide","Metalaxyl","Propiconazole",
    "Tebuconazole","Chlorothalonil","Captan","Myclobutanil","Azoxystrobin",
    "Trifloxystrobin","Hexaconazole","Abamectin","Dimethoate","Quinalphos",
    "Imidacloprid","Thiamethoxam","Bordeaux mixture","Tricyclazole",
    "Isoprothiolane","Fosetyl-Al","Cymoxanil","Spiromesifen",
]
_CHEMICAL_PRESERVE.sort(key=len, reverse=True)

def _protect_chemicals(text: str):
    placeholders = {}
    for i, name in enumerate(_CHEMICAL_PRESERVE):
        key = f"CHEM_{i}"
        pattern = re.compile(re.escape(name), re.IGNORECASE)
        if pattern.search(text):
            placeholders[key] = pattern.search(text).group(0)
            text = pattern.sub(key, text)
    return text, placeholders

def _restore_chemicals(text: str, placeholders: dict) -> str:
    for key, original in placeholders.items():
        text = text.replace(key, original)
    return text


@dataclass
class CropQueryResult:
    predictions:       List[Tuple[str, float]] = field(default_factory=list)
    resnet_hint:       Optional[str]           = None
    used_faiss:        bool                    = False
    faiss_similarity:  Optional[float]         = None
    rag_chunks:        list                    = field(default_factory=list)
    weather_ctx:       Optional[dict]          = None
    location_ctx:      Optional[dict]          = None
    irrigation_advice: str                     = ""
    english_advice:    str                     = ""
    translated_advice: str                     = ""
    target_language:   str                     = "English"

    @property
    def top_class(self) -> Optional[str]:
        return self.predictions[0][0] if self.predictions else None

    @property
    def top_confidence(self) -> Optional[float]:
        return self.predictions[0][1] if self.predictions else None

    @property
    def is_high_confidence(self) -> bool:
        return bool(self.top_confidence and self.top_confidence >= CONFIDENCE_GATE)

    @property
    def advice(self) -> str:
        return self.translated_advice or self.english_advice


class CropQueryPipeline:

    def __init__(self, skip_llava: bool = False):
        from .vision          import CropVisionModel
        from .retriever       import CropRetriever
        from .translator      import IndicTranslator
        from .context_fetcher import ContextFetcher

        print("[Pipeline] Initialising CropQuery...")
        self.vision          = CropVisionModel(device=BACKBONE_DEVICE)
        self.retriever       = CropRetriever()
        self.translator      = IndicTranslator()
        self.context_fetcher = ContextFetcher()
        self.advisor         = None

        if not skip_llava:
            from .advisor import LLaVAAdvisor
            self.advisor = LLaVAAdvisor()

        print("[Pipeline] Ready.")

    def run(
        self,
        image: Image.Image,
        farmer_question: str = "What should I do?",
        target_language: str = "Hindi",
    ) -> CropQueryResult:
        result = CropQueryResult(target_language=target_language)

        # Step 1: Vision — unpack tuple (FIXED: no _last_tensor side-channel)
        result.predictions, img_tensor = self.vision.predict(image)
        result.resnet_hint, result.used_faiss, result.faiss_similarity =             self.vision.gate(result.predictions, img_tensor)

        # Step 2: Context
        try:
            result.weather_ctx  = self.context_fetcher.get_weather()
            result.location_ctx = self.context_fetcher.get_location()
        except Exception as e:
            print(f"[Pipeline] Context fetch failed (non-fatal): {e}")

        # Step 3: Irrigation
        try:
            result.irrigation_advice = self.context_fetcher.get_irrigation_advice(
                crop=(result.top_class.split("__")[0].replace("_", " ")
                      if result.top_class else None),
                weather_ctx=result.weather_ctx,
            )
        except Exception as e:
            print(f"[Pipeline] Irrigation advisory failed (non-fatal): {e}")

        # Step 4: RAG
        result.rag_chunks = self.retriever.retrieve(
            predicted_class=result.top_class,
            farmer_question=farmer_question,
        )
        rag_context = self.retriever.build_rag_context(result.rag_chunks)

        # Step 5: LLaVA
        if self.advisor:
            result.english_advice = self.advisor.generate(
                image=image, rag_context=rag_context,
                farmer_question=farmer_question,
                resnet_hint=result.resnet_hint,
                confidence=result.top_confidence,
                weather_ctx=result.weather_ctx,
                location_ctx=result.location_ctx,
            )
        else:
            result.english_advice = (
                f"[RAG-only mode]\n\nTop prediction: "
                f"{result.top_class} ({result.top_confidence:.1f}%)\n\n"
                + rag_context
            )

        # Step 6: Translation
        from .advisor import _clean_jargon, _validate_advice
        if target_language != "English":
            protected, placeholders = _protect_chemicals(result.english_advice)
            translated = self.translator.translate(protected, target_language)
            result.translated_advice = _clean_jargon(
                _restore_chemicals(translated, placeholders)
            )
        else:
            result.english_advice = _validate_advice(
                result.english_advice, result.top_class
            )

        return result


Overwriting /kaggle/working/cropquery/pipeline.py


### 3i · app.py

**Fixes vs old version:**
- Removed unused `CropVisionModel` import
- `predict()` now returns `(preds, tensor)` tuple — updated accordingly
- `gr.Examples` only renders if images exist on disk (no crash)

In [12]:
%%writefile /kaggle/working/cropquery/app.py
"""
CropQuery — Gradio Demo
"""
import sys
sys.path.insert(0, "/kaggle/working")

import os
import gradio as gr
from PIL import Image
from typing import Optional

from cropquery import CropQueryPipeline
from cropquery.config import CONFIDENCE_GATE
from cropquery.translator import IndicTranslator

# pipeline is None until init() is called from the notebook.
# This prevents a second model load when Cell 11 does 'from cropquery.app import demo'.
pipeline: "Optional[CropQueryPipeline]" = None
LANGUAGES = IndicTranslator.available_languages()


def init(p: "CropQueryPipeline") -> None:
    """
    Bind an already-loaded pipeline to this module.
    Must be called from the notebook before demo.launch():

        from cropquery.app import demo, init
        init(pipeline)   # pipeline was loaded in Cell 6
        demo.queue()
        demo.launch(...)

    Raises RuntimeError if called when a pipeline is already bound,
    so a double-init (and accidental second GPU load) is caught early.
    """
    global pipeline
    if pipeline is not None:
        raise RuntimeError(
            "[app.init] Pipeline already initialised. "
            "Call init() exactly once per session."
        )
    pipeline = p
    print(f"[app] Pipeline bound — advisor={'loaded' if p.advisor else 'skipped'}")


def _confidence_bar(cls: str, conf: float) -> str:
    crop, cond = cls.split("__", 1) if "__" in cls else ("Unknown", cls)
    label  = f"{crop.replace('_', ' ')} — {cond.replace('_', ' ')}"
    colour = "#2e7d32" if conf >= CONFIDENCE_GATE else "#f57c00" if conf >= 50 else "#c62828"
    return (
        f"<div style='margin-bottom:10px'>"
        f"<div style='font-weight:600;margin-bottom:3px'>{label} "
        f"<span style='color:#888;font-size:13px'>({conf:.1f}%)</span></div>"
        f"<div style='background:#e0e0e0;border-radius:4px;height:8px;width:100%'>"
        f"<div style='background:{colour};border-radius:4px;height:8px;"
        f"width:{min(conf,100):.0f}%'></div></div></div>"
    )


def run_cropquery(
    image: Optional[Image.Image],
    farmer_question: str,
    target_language: str,
    confirmed_class: Optional[str],
):
    if image is None:
        yield (
            "<p style='color:#888'>Upload an image to start.</p>",
            "", gr.update(visible=False),
            gr.update(choices=[], value=None),
            "", gr.update(visible=False),
        )
        return

    # Step 1: Vision — unpack tuple (FIXED)
    preds, img_tensor = pipeline.vision.predict(image)
    resnet_hint, used_faiss, faiss_sim = pipeline.vision.gate(preds, img_tensor)

    is_high = preds and preds[0][1] >= CONFIDENCE_GATE
    badge = (
        "<span style='background:#2e7d32;color:white;padding:2px 8px;"
        "border-radius:4px;font-size:12px'>HIGH CONFIDENCE</span>"
        if is_high else
        "<span style='background:#f57c00;color:white;padding:2px 8px;"
        "border-radius:4px;font-size:12px'>LOW CONFIDENCE</span>"
    )
    bars_html    = "".join(_confidence_bar(cls, conf) for cls, conf in preds)
    disease_html = f"<div>{badge}"

    if used_faiss and faiss_sim is not None:
        disease_html += (
            f" <span style='background:#5c6bc0;color:white;padding:2px 8px;"
            f"border-radius:4px;font-size:12px'>"
            f"⚡ FAISS k-NN fallback (sim={faiss_sim:.2f})</span>"
        )
    disease_html += f"</div><br>{bars_html}"

    dropdown_choices = [f"{cls} ({conf:.0f}%)" for cls, conf in preds]

    # ── Low-confidence gate ───────────────────────────────────────────
    # Yield the confidence panel with a prompt, then STOP.
    # Advice is only generated after the farmer confirms via confirm_btn
    # → rerun_with_confirmed(). This prevents streaming unverified advice.
    if not is_high:
        yield (
            disease_html,
            "⚠️ Low confidence — please select the correct disease "
            "above and press ✅ Confirm to get advice.",
            gr.update(visible=True),
            gr.update(choices=dropdown_choices, value=dropdown_choices[0] if dropdown_choices else None),
            "", gr.update(visible=False),
        )
        return

    yield (
        disease_html, "⏳ Generating advice...",
        gr.update(visible=False),
        gr.update(choices=dropdown_choices, value=dropdown_choices[0] if dropdown_choices else None),
        "", gr.update(visible=False),
    )

    # Step 2: Context + RAG
    try:
        weather_ctx  = pipeline.context_fetcher.get_weather()
        location_ctx = pipeline.context_fetcher.get_location()
    except Exception:
        weather_ctx = location_ctx = None

    # Use confirmed_class if farmer selected one
    if confirmed_class:
        top_class   = confirmed_class.split(" (")[0]
        resnet_hint = top_class
    else:
        top_class = preds[0][0] if preds else None

    chunks      = pipeline.retriever.retrieve(top_class, farmer_question or "What should I do?")
    rag_context = pipeline.retriever.build_rag_context(chunks)

    # Step 3: Stream LLaVA
    try:
        streamer = pipeline.advisor.generate_streaming(
            image=image, rag_context=rag_context,
            farmer_question=farmer_question or "What should I do?",
            resnet_hint=resnet_hint,
            confidence=preds[0][1] if preds else None,
            weather_ctx=weather_ctx, location_ctx=location_ctx,
        )
        advice = ""
        for token in streamer:
            advice += token
            yield (
                disease_html, advice,
                gr.update(visible=not is_high),
                gr.update(choices=dropdown_choices, value=dropdown_choices[0] if dropdown_choices else None),
                "", gr.update(visible=False),
            )
    except Exception as e:
        advice = f"[Generation error: {e}]"
        yield (
            disease_html, advice,
            gr.update(visible=not is_high),
            gr.update(choices=dropdown_choices, value=dropdown_choices[0] if dropdown_choices else None),
            "", gr.update(visible=False),
        )

    # Step 4: Translate + irrigation
    if target_language != "English":
        advice = pipeline.translator.translate(advice, target_language)

    irrigation_text = ""
    try:
        irrigation_text = pipeline.context_fetcher.get_irrigation_advice(
            crop=top_class.split("__")[0].replace("_", " ") if top_class else None,
            weather_ctx=weather_ctx,
        )
        if target_language != "English" and irrigation_text:
            irrigation_text = pipeline.translator.translate(irrigation_text, target_language)
    except Exception:
        pass

    yield (
        disease_html, advice,
        gr.update(visible=not is_high),
        gr.update(choices=dropdown_choices, value=dropdown_choices[0] if dropdown_choices else None),
        irrigation_text, gr.update(visible=bool(irrigation_text)),
    )


def rerun_with_confirmed(
    image: Optional[Image.Image],
    farmer_question: str,
    target_language: str,
    confirmed_choice: str,
) -> str:
    if image is None or not confirmed_choice:
        return ""
    confirmed_class = confirmed_choice.split(" (")[0]
    try:
        weather_ctx  = pipeline.context_fetcher.get_weather()
        location_ctx = pipeline.context_fetcher.get_location()
    except Exception:
        weather_ctx = location_ctx = None
    rag_chunks  = pipeline.retriever.retrieve(confirmed_class, farmer_question)
    rag_context = pipeline.retriever.build_rag_context(rag_chunks)
    if pipeline.advisor:
        advice = pipeline.advisor.generate(
            image=image, rag_context=rag_context,
            farmer_question=farmer_question,
            resnet_hint=confirmed_class, confidence=100.0,
            weather_ctx=weather_ctx, location_ctx=location_ctx,
        )
    else:
        advice = f"[RAG-only]\n\n{rag_context}"
    if target_language != "English":
        advice = pipeline.translator.translate(advice, target_language)
    return advice


CSS = """
#title        { text-align: center; }
#disease-panel{ border: 1px solid #e0e0e0; border-radius: 8px; padding: 16px; }
#advice-panel { background: #f9fbe7; border-radius: 8px; padding: 16px; min-height: 200px; }
#low-conf-box { background: #fff3e0; border: 1px solid #f57c00; border-radius: 8px; padding: 16px; }
"""

with gr.Blocks(css=CSS, title="CropQuery") as demo:
    gr.HTML("""
    <div id='title'>
        <h1>🌾 CropQuery</h1>
        <p style='color:#555'>AI-Powered Crop Disease Detection & Farmer Advisory · Multilingual</p>
    </div>
    """)
    with gr.Row():
        with gr.Column(scale=1):
            image_input     = gr.Image(type="pil", label="📷 Upload leaf / crop image", height=300)
            question_input  = gr.Textbox(
                label="❓ Your question (optional)",
                placeholder="What spray should I use? / कौन सी दवाई लगाएं?",
                lines=2,
            )
            language_dropdown = gr.Dropdown(choices=LANGUAGES, value="Hindi", label="🌐 Output Language")
            run_btn         = gr.Button("🔍 Analyse", variant="primary", size="lg")
        with gr.Column(scale=2):
            disease_panel   = gr.HTML(
                value="<p style='color:#888'>Upload an image to start.</p>",
                elem_id="disease-panel",
            )
            with gr.Group(visible=False, elem_id="low-conf-box") as low_conf_panel:
                gr.Markdown("⚠️ **Low confidence** — please confirm which disease this looks like:")
                confirm_dropdown = gr.Dropdown(choices=[], label="Select the correct disease", interactive=True)
                confirm_btn      = gr.Button("✅ Confirm and get advice", variant="secondary")
            advice_output    = gr.Textbox(
                label="📋 Advice", lines=15, elem_id="advice-panel", show_copy_button=True
            )
            with gr.Group(visible=False) as irrigation_group:
                irrigation_output = gr.Textbox(label="💧 Irrigation Advisory", lines=3)

    run_btn.click(
        fn=run_cropquery,
        inputs=[image_input, question_input, language_dropdown, confirm_dropdown],
        outputs=[disease_panel, advice_output, low_conf_panel, confirm_dropdown, irrigation_output, irrigation_group],
    )
    confirm_btn.click(
        fn=rerun_with_confirmed,
        inputs=[image_input, question_input, language_dropdown, confirm_dropdown],
        outputs=[advice_output],
    )

    _example_candidates = [
        ["examples/tomato_late_blight.jpg", "What spray should I use?", "Hindi"],
        ["examples/wheat_rust.jpg",         "How serious is this?",     "Telugu"],
        ["examples/corn_gray_leaf.jpg",     "",                          "English"],
    ]
    _examples = [e for e in _example_candidates if os.path.exists(e[0])]
    if _examples:
        gr.Examples(examples=_examples, inputs=[image_input, question_input, language_dropdown])

    gr.HTML("""
    <div style='text-align:center;color:#888;margin-top:16px;font-size:12px'>
        Ensemble Vision · LLaVA-1.6-7B · ChromaDB RAG · deep-translator
    </div>
    """)

if __name__ == "__main__":
    demo.queue()
    demo.launch(share=True, server_port=7860, show_error=True)


Overwriting /kaggle/working/cropquery/app.py


### 3j · \_\_init\_\_.py

**Fix vs old version:** Docstring updated to reflect actual architecture.

In [13]:
%%writefile /kaggle/working/cropquery/__init__.py
"""
CropQuery — AI-Powered Crop Disease Detection & Farmer Advisory System

Architecture
------------
- Vision:       3-backbone ensemble (ResNet50 + EfficientNet-B3 + DenseNet121)
                + FAISS k-NN fallback for low-confidence cases
- Advisory:     LLaVA-1.6-Mistral-7B (4-bit quantised, cuda:1)
- Knowledge:    ChromaDB RAG — AgroBench entries + Kisan Call Centre Q&A
- Translation:  deep-translator (GoogleTranslator) — 11 Indian languages
- Context:      Open-Meteo weather + IP geolocation + irrigation advisory

Quick start
-----------
    from cropquery import CropQueryPipeline
    from PIL import Image

    pipeline = CropQueryPipeline()
    result   = pipeline.run(
        image           = Image.open("leaf.jpg"),
        farmer_question = "What should I spray?",
        target_language = "Hindi",
    )
    print(result.advice)
"""

from .pipeline        import CropQueryPipeline, CropQueryResult
from .context_fetcher import ContextFetcher
from .vision          import CropVisionModel
from .retriever       import CropRetriever
from .translator      import IndicTranslator
from .knowledge_base  import KnowledgeBaseBuilder
from .advisor         import LLaVAAdvisor
from . import config

__all__ = [
    "CropQueryPipeline",
    "CropQueryResult",
    "CropVisionModel",
    "CropRetriever",
    "IndicTranslator",
    "KnowledgeBaseBuilder",
    "LLaVAAdvisor",
    "config",
]


Overwriting /kaggle/working/cropquery/__init__.py


## 4 · Verify paths + write class_names.json

In [14]:
import sys, os, torch, json, gc
sys.path.insert(0, "/kaggle/working")
import cropquery.config as cfg

# Paths are already correct in config.py — nothing to patch.
# Just verify all required files exist.
ckpt = torch.load(cfg.RESNET_CKPT, map_location="cpu")
class_to_idx = ckpt["class_to_idx"]
idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names  = [idx_to_class[i] for i in range(len(idx_to_class))]
cfg.NUM_CLASSES = len(class_names)

with open(cfg.CLASS_NAMES_PATH, "w") as f:
    json.dump(class_names, f, indent=2)

print(f"class_names.json written — {len(class_names)} classes")
print("Sample:", class_names[:5])

del ckpt; gc.collect(); torch.cuda.empty_cache()

required = [
    cfg.RESNET_CKPT, cfg.EFFICIENTNET_CKPT, cfg.DENSENET_CKPT,
    cfg.BANK_FEATURES, cfg.BANK_LABELS, cfg.CLASS_NAMES_PATH,
]
all_ok = True
for path in required:
    status = "✓" if os.path.exists(path) else "✗ MISSING"
    if "MISSING" in status: all_ok = False
    print(f"  {status}  {path}")
print("\n✅ All files ready." if all_ok else "\n❌ Fix missing files before continuing.")


class_names.json written — 64 classes
Sample: ['Apple__Apple_Scab', 'Apple__Black_Rot', 'Apple__Cedar_Apple_Rust', 'Apple__Healthy', 'Bell_Pepper__Bacterial_Spot']
  ✓  /kaggle/input/datasets/vitthaljauhari2/cropquerydataset/resnet50_best.pth
  ✓  /kaggle/input/datasets/vitthaljauhari2/cropquerydataset/efficientnet_best.pth
  ✓  /kaggle/input/datasets/vitthaljauhari2/cropquerydataset/densenet_best.pth
  ✓  /kaggle/input/datasets/vitthaljauhari2/cropquerydataset/feature_bank_features.npy
  ✓  /kaggle/input/datasets/vitthaljauhari2/cropquerydataset/feature_bank_labels.npy
  ✓  /kaggle/working/cropquery/class_names.json

✅ All files ready.


## 5 · Build knowledge base

Skipped automatically if already built. Pass `force_rebuild=True` to rebuild from scratch.

In [15]:
from datasets import load_dataset
import os

ds = load_dataset("vitthaljauhari/agrobench", split="train", token=os.environ["HF_TOKEN"])
print(f"Loaded {len(ds)} entries")
print(ds[0])  # should show crop, condition, text fields

agrobench.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 31 entries
{'crop': 'Tomato', 'condition': 'Late Blight', 'text': 'Tomato Late Blight (Phytophthora infestans). Symptoms: Water-soaked lesions on leaves that turn brown with a pale green border; white sporulation on undersides; fruit shows greasy dark patches. Severity: High — can destroy an entire crop in 7-10 days under humid conditions. Organic control: Copper-based fungicide (Bordeaux mixture) spray every 7 days. Chemical control: Mancozeb 75 WP (2g/L) or Metalaxyl + Mancozeb. Prevention: Avoid overhead irrigation; maintain plant spacing for air circulation; destroy infected debris; use certified disease-free seeds.'}


In [16]:
from cropquery.knowledge_base import KnowledgeBaseBuilder

builder = KnowledgeBaseBuilder()
count   = builder.build(force_rebuild=False)
print(f"Knowledge base ready — {count} documents")
print(builder.stats())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[KB] Collection already has 109888 docs. Pass force_rebuild=True to overwrite.
Knowledge base ready — 109888 documents
{'total_documents': 109888, 'source_breakdown': {'kisan_call_centre': 1000}}


## 6 · Load pipeline

This loads all three backbones and LLaVA. Takes ~2-3 min on first run.

In [17]:
# ── Free GPU memory before loading pipeline ──────────────────────────
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
import gc

# Confirm device assignment before anything loads
print("cuda:0 free:", round(torch.cuda.mem_get_info(0)[0]/1e9, 2), "GB")
print("cuda:1 free:", round(torch.cuda.mem_get_info(1)[0]/1e9, 2), "GB")

cuda:0 free: 15.53 GB
cuda:1 free: 15.53 GB


In [18]:
import sys
sys.path.insert(0, "/kaggle/working")

from cropquery import CropQueryPipeline
from cropquery.config import CONFIDENCE_GATE
from cropquery.translator import IndicTranslator

pipeline  = CropQueryPipeline(skip_llava=False)
LANGUAGES = IndicTranslator.available_languages()
print("Pipeline loaded. LANGUAGES:", LANGUAGES)


[Pipeline] Initialising CropQuery...
[Vision] Loaded 3 backbones on cuda:0
[FAISS] Loaded index: 113011 vectors, dim=4608, k=7
[RAG] Connected to ChromaDB. 109888 documents.


The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


[LLaVA] GPU 1 free before load: 15.64 GiB
[LLaVA] Loading /kaggle/working/llava_cache (4-bit)...


Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

[LLaVA] Model loaded.
[Pipeline] Ready.
Pipeline loaded. LANGUAGES: ['English', 'Hindi', 'Tamil', 'Telugu', 'Kannada', 'Malayalam', 'Bengali', 'Marathi', 'Gujarati', 'Punjabi', 'Odia', 'Urdu']


## 7 · FastAPI app

Imports cleanly from the installed cropquery module — no `__main__` scope games.

In [19]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from PIL import Image
import io

app = FastAPI()

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/diagnose")
async def diagnose(
    image:           UploadFile = File(...),
    farmer_question: str        = Form(default="What should I do?"),
    target_language: str        = Form(default="English"),
    confirmed_class: str        = Form(default=None),
):
    img_bytes = await image.read()
    pil_image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    confirmed = confirmed_class if confirmed_class and confirmed_class.strip() else None

    # Vision — predict() returns list, tensor stored as _last_tensor side-effect
    preds, img_tensor = pipeline.vision.predict(pil_image)
    resnet_hint, used_faiss, faiss_sim = pipeline.vision.gate(preds, img_tensor)

    if confirmed:
        resnet_hint = confirmed.split(" (")[0]

    top_class  = resnet_hint or (preds[0][0] if preds else None)
    confidence = preds[0][1] if preds else None

    # Context
    try:
        weather_ctx  = pipeline.context_fetcher.get_weather()
        location_ctx = pipeline.context_fetcher.get_location()
    except Exception:
        weather_ctx = location_ctx = None

    # RAG
    chunks      = pipeline.retriever.retrieve(top_class, farmer_question or "What should I do?")
    rag_context = pipeline.retriever.build_rag_context(chunks)

    # LLaVA
    if pipeline.advisor:
        advice = pipeline.advisor.generate(
            image=pil_image,
            rag_context=rag_context,
            farmer_question=farmer_question or "What should I do?",
            resnet_hint=resnet_hint,
            confidence=confidence,
            weather_ctx=weather_ctx,
            location_ctx=location_ctx,
        )
    else:
        advice = f"[RAG-only]\n\n{rag_context}"

    # Translation
    if target_language != "English":
        advice = pipeline.translator.translate(advice, target_language)

    # Irrigation
    irrigation_text = ""
    try:
        crop_name = top_class.split("__")[0].replace("_", " ") if top_class else None
        irrigation_text = pipeline.context_fetcher.get_irrigation_advice(
            crop=crop_name,
            weather_ctx=weather_ctx,
        )
        if target_language != "English" and irrigation_text:
            irrigation_text = pipeline.translator.translate(irrigation_text, target_language)
    except Exception:
        pass

    return JSONResponse({
        "advice":          advice,
        "irrigation_text": irrigation_text,
        "top_class":       top_class,
        "confidence":      float(confidence) if confidence is not None else None,
    })

print("FastAPI app defined. Run next cell to start server.")

FastAPI app defined. Run next cell to start server.


## 8 · Start uvicorn server

In [20]:
import nest_asyncio, threading, time, uvicorn, os
nest_asyncio.apply()

# Kill any previous server on port 8000
os.system("fuser -k 8000/tcp 2>/dev/null")
time.sleep(1)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(2)
print("Server started on port 8000")

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server started on port 8000


## 9 · Start ngrok tunnel

In [21]:
from pyngrok import conf, ngrok
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("NGROK_TOKEN")
conf.get_default().auth_token = token

tunnel   = ngrok.connect(8000)
base_url = tunnel.public_url
print(f"Base URL:          {base_url}")
print(f"Health check:      {base_url}/health")
print(f"Diagnose endpoint: {base_url}/diagnose")

Base URL:          https://c8a3-34-11-10-207.ngrok-free.app                                         
Health check:      https://c8a3-34-11-10-207.ngrok-free.app/health
Diagnose endpoint: https://c8a3-34-11-10-207.ngrok-free.app/diagnose


## 10 · Smoke test

Hits `/health` first, then `/diagnose` with a real image.

In [22]:
import requests

# Health check
r = requests.get(f"{base_url}/health")
print("Health:", r.json())

# Diagnose — swap path for any leaf image available on disk
_test_image = (
    "/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated/"
    "Grape/Test/Leaf Blight/"
    "19278e7d-ac39-4bd4-a04f-f04014adbd7e___FAM_L.Blight 1486_flipLR.JPG"
)

if os.path.exists(_test_image):
    with open(_test_image, "rb") as f:
        r = requests.post(
            f"{base_url}/diagnose",
            files={"image": f},
            data={"farmer_question": "What is wrong with my crop?"},
        )
    print("Status:", r.status_code)
    result = r.json()
    print("Advice preview:", result.get("advice", "")[:300])
else:
    print("Test image not found — update _test_image path above")

INFO:     34.11.10.207:0 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok'}
[Vision] FAISS fallback → Grape__Leaf_Blight (sim=0.953)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[RAG] Warning: using unfiltered fallback — verify chunks are relevant


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[LLaVA] Visual description: - Colour of affected areas: yellow
- Location on leaf: edges
- Spread: small patches...
INFO:     34.11.10.207:0 - "POST /diagnose HTTP/1.1" 200 OK
Status: 200
Advice preview: ❓ आपका जवाब (Your Answer)
* Leaf blight
* [High]
* Yellow spots on edges of leaves

🌿 समस्या (Problem)
* Leaf blight
* [Medium]
* Yellow spots on edges of leaves

⚠️ स्थिति (Severity)
* Mild
* Damaging the leaves, affecting photosynthesis and overall growth

✅ क्या करें (What to do now)
* Organic: R


## 11 · Launch Gradio UI

Run this cell last. LLaVA must already be loaded (pipeline loaded in cell 6).

In [23]:
import sys
sys.path.insert(0, "/kaggle/working")

from cropquery.app import demo, init
init(pipeline)   

demo.queue()
demo.launch(share=True, server_port=7860, show_error=True)

[app] Pipeline bound — advisor=loaded
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0b8e10c09e5a1a4585.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[Vision] FAISS fallback → Cassava__Mosaic_Disease (sim=0.753)
[Vision] FAISS fallback → Cashew__Healthy (sim=0.781)
[Vision] FAISS fallback → Cashew__Anthracnose (sim=0.760)
[Vision] FAISS fallback → Cassava__Mosaic_Disease (sim=0.775)
[Vision] FAISS fallback → Cashew__Anthracnose (sim=0.755)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[RAG] Warning: using unfiltered fallback — verify chunks are relevant


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[LLaVA] Visual description: - Colour of affected areas: yellow
- Location on leaf: edges
- Spread: small patches...
[Vision] FAISS fallback → Tomato__Late_Blight (sim=0.727)
